# 2. Extract Dataset

Extract conditioning factor values from raster layers at each sample point location.

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path

## Configuration / 

Adjust the paths below to match your local raster file locations.

In [ ]:
# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "samples"

# Sample point (output from Notebook 1)
SAMPLE_FLOOD = OUTPUT_DIR / 'sample_points.csv'

# Raster layers for each conditioning factor
RASTERS = {
    'elevation': 'data/raw/elevation.tif',
    'slope': 'data/raw/slope.tif',
    'distance_to_river': 'data/raw/distance_to_river.tif',
    'distance_to_coast': 'data/raw/distance_to_coast.tif',
    'land_cover': 'data/raw/land_cover.tif',
    'soil_type': 'data/raw/soil_type.tif',
    'ndvi': 'data/raw/ndvi.tif',
    'rainfall': 'data/raw/rainfall.tif',
}


## Load Sample Points / 

In [ ]:
all_points = pd.read_csv(SAMPLE_FLOOD)
print(f"Total points: {len(all_points)}")
print(f"Columns: {list(all_points.columns)}")

label_candidates = ["label", "class", "kelas", "Label", "Class"]
label_col = next((c for c in label_candidates if c in all_points.columns), None)

if label_col is None:
    raise ValueError(
        "Kolom label tidak ditemukan. "
        "Pastikan sample_points.csv memiliki kolom label (1=flood, 0=non-flood)."
    )

all_points["label"] = all_points[label_col].astype(int)

n_flood    = int((all_points["label"] == 1).sum())
n_nonflood = int((all_points["label"] == 0).sum())
print(f"Flood: {n_flood}, Non-flood: {n_nonflood}")


## Extract Raster Values / 

In [ ]:
coords = list(zip(all_points['lon'], all_points['lat']))

for param, path in RASTERS.items():
    with rasterio.open(path) as src:
        all_points[param] = [v[0] for v in src.sample(coords)]
    print(f"Extracted: {param}")

missing = all_points.isna().sum()
if (missing > 0).any():
    print("WARNING - missing values detected:")
    print(missing[missing > 0])
else:
    print("No missing values.")

## Save Training Dataset / 

In [ ]:
OUTPUT_CSV = OUTPUT_DIR / 'dataset_training.csv'
df_out = all_points.drop(columns=['lon', 'lat'])
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Saved dataset: {len(df_out)} rows, {len(df_out.columns)} columns')
print(f'Columns: {list(df_out.columns)}')

df_out.head(10)